# Polyp Detection — Data Preparation

Downloads **Kvasir-SEG** and **CVC-ClinicDB**, converts segmentation masks to YOLO bounding boxes,
and creates a unified dataset ready for YOLOv8 training.

**Run in Google Colab only.**

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/repos/deep-learning')

from src.config.paths import DRIVE, LANDING, BRONZE_DETECTION, TRAINED
print(f'Drive root: {DRIVE}')
print(f'Landing: {LANDING}')
print(f'Bronze Detection: {BRONZE_DETECTION}')

In [ ]:
# Cell 2: Install dependencies
!pip install -q gdown opencv-python-headless

In [ ]:
# Cell 3: Download Kvasir-SEG (1000 images + masks)
import os
import zipfile
from pathlib import Path

WORK_DIR = Path('/content/polyp_data')
WORK_DIR.mkdir(exist_ok=True)

# Download Kvasir-SEG from official source
kvasir_zip = WORK_DIR / 'Kvasir-SEG.zip'
if not kvasir_zip.exists():
    !wget -q -O {kvasir_zip} 'https://datasets.simula.no/downloads/kvasir-seg.zip'
    print('Downloaded Kvasir-SEG')

kvasir_dir = WORK_DIR / 'Kvasir-SEG'
if not kvasir_dir.exists():
    with zipfile.ZipFile(kvasir_zip, 'r') as z:
        z.extractall(WORK_DIR)
    print('Extracted Kvasir-SEG')

# List extracted contents
for p in sorted(kvasir_dir.iterdir()):
    if p.is_dir():
        print(f'  {p.name}/  ({len(list(p.iterdir()))} files)')
    else:
        print(f'  {p.name}')

In [ ]:
# Cell 4: Download CVC-ClinicDB (612 images + masks)
import gdown

cvc_zip = WORK_DIR / 'CVC-ClinicDB.zip'
if not cvc_zip.exists():
    # CVC-ClinicDB from Google Drive (commonly hosted)
    gdown.download(
        'https://drive.google.com/uc?id=1o8OfBvYE6K-EpDyvzsmMPndnUMwb540R',
        str(cvc_zip), quiet=False
    )
    print('Downloaded CVC-ClinicDB')

cvc_dir = WORK_DIR / 'CVC-ClinicDB'
if not cvc_dir.exists():
    with zipfile.ZipFile(cvc_zip, 'r') as z:
        z.extractall(WORK_DIR)
    print('Extracted CVC-ClinicDB')

# List structure
for p in sorted(WORK_DIR.iterdir()):
    if p.is_dir():
        count = len(list(p.rglob('*.png'))) + len(list(p.rglob('*.jpg')))
        print(f'  {p.name}/  ({count} images)')

In [ ]:
# Cell 5: Convert segmentation masks to YOLO bounding boxes
import cv2
import numpy as np
import shutil

def mask_to_yolo_bbox(mask_path: Path) -> list:
    """
    Convert a binary segmentation mask to YOLO format bounding box(es).
    Returns list of [class_id, x_center, y_center, width, height] (normalized).
    If multiple polyps are close, they get merged into one box.
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    
    h, w = mask.shape
    
    # Threshold to binary
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return []
    
    # Get bounding boxes for all contours
    boxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 100:  # Skip tiny noise regions
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        boxes.append([x, y, x + bw, y + bh])
    
    if not boxes:
        return []
    
    # Merge nearby boxes (IoU-based or proximity)
    merged = merge_nearby_boxes(boxes, distance_threshold=50)
    
    # Convert to YOLO format
    yolo_labels = []
    for (x1, y1, x2, y2) in merged:
        x_center = ((x1 + x2) / 2) / w
        y_center = ((y1 + y2) / 2) / h
        bw_norm = (x2 - x1) / w
        bh_norm = (y2 - y1) / h
        yolo_labels.append([0, x_center, y_center, bw_norm, bh_norm])
    
    return yolo_labels


def merge_nearby_boxes(boxes, distance_threshold=50):
    """
    Merge boxes that are close to each other.
    Uses iterative merging: if two boxes overlap or are within
    distance_threshold pixels, merge them.
    """
    if len(boxes) <= 1:
        return boxes
    
    merged = list(boxes)
    changed = True
    
    while changed:
        changed = False
        new_merged = []
        used = set()
        
        for i in range(len(merged)):
            if i in used:
                continue
            current = list(merged[i])
            
            for j in range(i + 1, len(merged)):
                if j in used:
                    continue
                other = merged[j]
                
                # Check if boxes are close enough to merge
                if (current[0] - distance_threshold <= other[2] and
                    current[2] + distance_threshold >= other[0] and
                    current[1] - distance_threshold <= other[3] and
                    current[3] + distance_threshold >= other[1]):
                    # Merge: take bounding box of both
                    current = [
                        min(current[0], other[0]),
                        min(current[1], other[1]),
                        max(current[2], other[2]),
                        max(current[3], other[3]),
                    ]
                    used.add(j)
                    changed = True
            
            new_merged.append(current)
            used.add(i)
        
        merged = new_merged
    
    return merged


# Test on a sample
sample_masks = list((kvasir_dir / 'masks').glob('*.jpg'))
if sample_masks:
    test_result = mask_to_yolo_bbox(sample_masks[0])
    print(f'Sample mask: {sample_masks[0].name}')
    print(f'YOLO labels: {test_result}')
else:
    print('Checking for PNG masks...')
    sample_masks = list((kvasir_dir / 'masks').glob('*.png'))
    if sample_masks:
        test_result = mask_to_yolo_bbox(sample_masks[0])
        print(f'Sample mask: {sample_masks[0].name}')
        print(f'YOLO labels: {test_result}')

In [ ]:
# Cell 6: Build YOLO dataset structure
# Structure: polyp_yolo/
#   images/train/  images/val/  images/test/
#   labels/train/  labels/val/  labels/test/

import random
random.seed(42)

YOLO_DIR = BRONZE_DETECTION / 'polyp_yolo'
for split in ['train', 'val', 'test']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print(f'YOLO dataset dir: {YOLO_DIR}')


def find_image_mask_pairs(dataset_name, img_dir, mask_dir):
    """Find matching image-mask pairs from a dataset."""
    pairs = []
    img_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif'}
    
    images = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in img_extensions])
    masks = sorted([f for f in mask_dir.iterdir() if f.suffix.lower() in img_extensions])
    
    # Build mask lookup by stem
    mask_lookup = {m.stem: m for m in masks}
    
    for img in images:
        mask = mask_lookup.get(img.stem)
        if mask:
            pairs.append((dataset_name, img, mask))
    
    print(f'  {dataset_name}: {len(pairs)} image-mask pairs found')
    return pairs


# Collect all pairs
all_pairs = []

# Kvasir-SEG
kvasir_imgs = kvasir_dir / 'images'
kvasir_masks = kvasir_dir / 'masks'
if kvasir_imgs.exists() and kvasir_masks.exists():
    all_pairs.extend(find_image_mask_pairs('kvasir', kvasir_imgs, kvasir_masks))

# CVC-ClinicDB — structure varies, auto-detect
cvc_candidates = [
    (cvc_dir / 'Original', cvc_dir / 'Ground Truth'),
    (cvc_dir / 'images', cvc_dir / 'masks'),
    (WORK_DIR / 'CVC-612' / 'Original', WORK_DIR / 'CVC-612' / 'Ground Truth'),
]
for img_d, mask_d in cvc_candidates:
    if img_d.exists() and mask_d.exists():
        all_pairs.extend(find_image_mask_pairs('cvc', img_d, mask_d))
        break

# If CVC wasn't found in expected locations, search for it
if not any(name == 'cvc' for name, _, _ in all_pairs):
    print('\nSearching for CVC-ClinicDB images...')
    for d in WORK_DIR.rglob('*'):
        if d.is_dir() and d.name.lower() in ('original', 'images'):
            parent = d.parent
            for mask_name in ('Ground Truth', 'masks', 'ground_truth', 'GT'):
                mask_d = parent / mask_name
                if mask_d.exists():
                    print(f'  Found: {d} + {mask_d}')
                    all_pairs.extend(find_image_mask_pairs('cvc', d, mask_d))
                    break

print(f'\nTotal pairs: {len(all_pairs)}')

In [ ]:
# Cell 7: Split and convert all pairs
random.shuffle(all_pairs)

n = len(all_pairs)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

splits = {
    'train': all_pairs[:n_train],
    'val': all_pairs[n_train:n_train + n_val],
    'test': all_pairs[n_train + n_val:],
}

stats = {'train': 0, 'val': 0, 'test': 0}
skipped = 0

for split_name, pairs in splits.items():
    for dataset_name, img_path, mask_path in pairs:
        # Convert mask to YOLO labels
        labels = mask_to_yolo_bbox(mask_path)
        if not labels:
            skipped += 1
            continue
        
        # Unique filename to avoid collisions
        unique_name = f'{dataset_name}_{img_path.stem}'
        
        # Copy image (convert to jpg if needed)
        dst_img = YOLO_DIR / 'images' / split_name / f'{unique_name}.jpg'
        if not dst_img.exists():
            img = cv2.imread(str(img_path))
            if img is not None:
                cv2.imwrite(str(dst_img), img)
        
        # Write YOLO label file
        dst_label = YOLO_DIR / 'labels' / split_name / f'{unique_name}.txt'
        with open(dst_label, 'w') as f:
            for lbl in labels:
                f.write(' '.join(f'{v:.6f}' if i > 0 else str(int(v)) for i, v in enumerate(lbl)))
                f.write('\n')
        
        stats[split_name] += 1

print(f'\nDataset created at: {YOLO_DIR}')
print(f'  Train: {stats["train"]} images')
print(f'  Val:   {stats["val"]} images')
print(f'  Test:  {stats["test"]} images')
print(f'  Skipped (empty masks): {skipped}')

In [ ]:
# Cell 8: Create YOLO dataset YAML config
import yaml

dataset_yaml = {
    'path': str(YOLO_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': ['polyp'],
}

yaml_path = YOLO_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f'Dataset YAML written to: {yaml_path}')
print()
with open(yaml_path) as f:
    print(f.read())

In [ ]:
# Cell 9: Visualize a few samples with bounding boxes
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_yolo_sample(img_path, label_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img)
    
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                xc, yc, bw, bh = [float(x) for x in parts[1:]]
                
                # Convert to pixel coords
                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                box_w = bw * w
                box_h = bh * h
                
                rect = patches.Rectangle(
                    (x1, y1), box_w, box_h,
                    linewidth=3, edgecolor='lime', facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, 'polyp', color='lime', fontsize=12,
                        fontweight='bold', backgroundcolor='black')
    
    ax.set_title(img_path.stem)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Show 6 random training samples
train_images = sorted((YOLO_DIR / 'images' / 'train').glob('*.jpg'))
samples = random.sample(train_images, min(6, len(train_images)))

for img_p in samples:
    label_p = YOLO_DIR / 'labels' / 'train' / f'{img_p.stem}.txt'
    visualize_yolo_sample(img_p, label_p)

In [ ]:
# Cell 10: Copy your custom polyp images (no labels — for inference later)
custom_dir = LANDING / 'polyp'
custom_dest = YOLO_DIR / 'custom_unlabeled'
custom_dest.mkdir(exist_ok=True)

if custom_dir.exists():
    custom_images = list(custom_dir.glob('*.jpg')) + list(custom_dir.glob('*.png'))
    for img in custom_images:
        shutil.copy2(img, custom_dest / img.name)
    print(f'Copied {len(custom_images)} custom polyp images to {custom_dest}')
else:
    print(f'Custom polyp dir not found: {custom_dir}')

# Cleanup temp downloads
print('\n✅ Data preparation complete!')
print(f'Dataset: {YOLO_DIR}')
print(f'Config:  {yaml_path}')